# Homework: Agentic RAG

## Question 1: How many lesson pages

> How many lesson pages are in the dataset?
>
> * 24
> * 72
> * 240
> * 720

The instructions give the code below that can traverse files in a Github repo:

In [16]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

With this code it's simple to find the number of pages:

In [17]:
files = reader.read()
print("Q1. How many lesson pages?")
print("Answer: {}".format(len(files)))  

Q1. How many lesson pages?
Answer: 72


## Question 2: Indexing and searching


> Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:
>
>>     How does the agentic loop keep calling the model until it stops?
>
> What's the filename of the first result?
>
>    * 01-agentic-rag/lessons/03-rag.md
>    * 01-agentic-rag/lessons/14-agentic-loop.md
>    * 04-evaluation/lessons/13-llm-as-judge.md
>    * 06-best-practices/lessons/02-hybrid-search.md


In [18]:
# Prepare the lesson markdown documents for indexing
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [19]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

results = index.search("How does the agentic loop keep calling the model until it stops?")

print("Q2: What's the filename of the first result?")
print("Answer: {}".format(results[0]["filename"]))

Q2: What's the filename of the first result?
Answer: 01-agentic-rag/lessons/14-agentic-loop.md


## Question 3: RAG

> Build a RAG over the index from Q2 and answer the query:
>
>>     How does the agentic loop keep calling the model until it stops?
>
> Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?
>
>   * 700
>   * 7000
>   * 70000
>   * 700000


I modified the provided `RAGBase` class from [rag_helper.py](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/code/rag_helper.py) slightly:
* Renamed to ZoomCampRAG
* Requires the `INSTRUCTION` and `PROMPT_TEMPLATE` to be passed into the constructor
* Renamed method `rag` to `query` and 'privatized' other methods by prefixing with `__` since only `query` seems like it should be externally callable
* Commented out the `boost_dict` and `filter_dict` because they require the names of the index data fields and with only `filename` and `content` there isn't likely much benefit to boosting.

Other changes that might be useful:

Allow `boost_dict` and `filter_dict` to be passed in to the `query` method because they're more closely tied with the query string than to the RAG helper itself.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import ZoomCampRAG

load_dotenv()
openai_client = OpenAI()


INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

question3 = "How does the agentic loop keep calling the model until it stops?"

rag = ZoomCampRAG(index, openai_client, INSTRUCTIONS, PROMPT_TEMPLATE)

output_text, usage = rag.query(question3)

print("Q3: Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?")
print("Answer: {}".format(usage.input_tokens))

ImportError: cannot import name 'ZoomCampRAG' from 'ZoomCampRAG' (/workspaces/llm-zoomcamp-2026/01-agentic-rag/homework/ZoomCampRAG.py)

Q4: How many chunks do you get?

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print ("Q4: How many chunks do you get?")
print ("Answer: {}".format(len(chunks)))

Q5. RAG with chunking
Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

    about the same
    3× fewer
    10× fewer
    30× fewer

In [ ]:
chunked_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunked_index.fit(chunks)
myrag2 = RAGBase(chunked_index, openai_client)
output_text, chunked_usage = myrag2.rag(Question)
print(output_text)
print("\n\nQ5: Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?")
print("Answer: {}".format(round(usage.input_tokens / chunked_usage.input_tokens)))

How many times did the agent call search?

    Note: the agent decides this itself, so it varies a little between runs - pick the closest option. We measured this with OpenAI gpt-5.4-mini; with a different model or provider the number may differ, so keep that in mind.

    0
    4
    10
    20


In [ ]:
print("Q6: How many times did the agent call search?")
print("Answer: {}".format(-1))